In [1]:
import pandas as pd
import numpy as np

* Loading and examining the data

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
store = pd.read_csv('store.csv')

/tmp/ipykernel_548/2817523562.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('train.csv')


In [3]:
print('Train Shape:', train.shape)
print('Test Shape:', test.shape)
print('Store Shape:', store.shape)

Train Shape: (1017209, 9)
Test Shape: (41088, 8)
Store Shape: (1115, 10)


In [4]:
print("Train columns:")
for i, col in enumerate(train.columns):
    print(i, col, train[col].dtype)

print("\nTrain sample:")
display(train.head())

print("\nStore sample:")
display(store.head())

print("\nTest sample:")
display(test.head())

Train columns:
0 Store int64
1 DayOfWeek int64
2 Date object
3 Sales int64
4 Customers int64
5 Open int64
6 Promo int64
7 StateHoliday object
8 SchoolHoliday int64

Train sample:


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1



Store sample:


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN



Test sample:


,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
0,1,1,4,2015-09-17,1.0,1,0,0
1,2,3,4,2015-09-17,1.0,1,0,0
2,3,7,4,2015-09-17,1.0,1,0,0
3,4,8,4,2015-09-17,1.0,1,0,0
4,5,9,4,2015-09-17,1.0,1,0,0


In [5]:
print("Missing Store IDs:")
print("Train:", train["Store"].isna().sum())
print("Test:", test["Store"].isna().sum())
print("Store:", store["Store"].isna().sum())

print("\nDuplicate Store IDs in store.csv:")
print(store["Store"].duplicated().sum())

print("\nUnique Store IDs:")
print("Train:", train["Store"].nunique())
print("Test:", test["Store"].nunique())
print("Store:", store["Store"].nunique())

Missing Store IDs:
Train: 0
Test: 0
Store: 0

Duplicate Store IDs in store.csv:
0

Unique Store IDs:
Train: 1115
Test: 856
Store: 1115


In [6]:
train_rows_before = len(train)
test_rows_before = len(test)

# Merge store-level features into training data
train_merged = train.merge(
    store,
    on="Store",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Training merge status:")
print(train_merged["_merge"].value_counts())

assert len(train_merged) == train_rows_before
assert (train_merged["_merge"] == "both").all()

train_merged.drop(columns="_merge", inplace=True)


# Merge store-level features into test data
test_merged = test.merge(
    store,
    on="Store",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("\nTest merge status:")
print(test_merged["_merge"].value_counts())

assert len(test_merged) == test_rows_before
assert (test_merged["_merge"] == "both").all()

test_merged.drop(columns="_merge", inplace=True)


# Final verification
print("\nMerged shapes:")
print("Train:", train_merged.shape)
print("Test:", test_merged.shape)

Training merge status:
_merge
both          1017209
left_only           0
right_only          0
Name: count, dtype: int64

Test merge status:
_merge
both          41088
left_only         0
right_only        0
Name: count, dtype: int64

Merged shapes:
Train: (1017209, 18)
Test: (41088, 17)


In [7]:
print("TRAIN MISSING VALUES")
train_missing = (
    train_merged.isnull()
    .sum()
    .sort_values(ascending=False)
)

display(train_missing[train_missing > 0])

print("\nTEST MISSING VALUES")
test_missing = (
    test_merged.isnull()
    .sum()
    .sort_values(ascending=False)
)

display(test_missing[test_missing > 0])

TRAIN MISSING VALUES


,0
Promo2SinceWeek,508031
PromoInterval,508031
Promo2SinceYear,508031
CompetitionOpenSinceYear,323348
CompetitionOpenSinceMonth,323348
CompetitionDistance,2642



TEST MISSING VALUES


,0
PromoInterval,17232
Promo2SinceYear,17232
Promo2SinceWeek,17232
CompetitionOpenSinceMonth,15216
CompetitionOpenSinceYear,15216
CompetitionDistance,96
Open,11


In [8]:
print("Promo2 distribution in training:")
print(train_merged["Promo2"].value_counts(dropna=False))

print("\nPromo2 missingness check in training:")
display(
    train_merged.groupby("Promo2", dropna=False)[
        ["Promo2SinceWeek", "Promo2SinceYear", "PromoInterval"]
    ].agg(lambda col: col.isna().sum())
)

print("\nPromo2 distribution in test:")
print(test_merged["Promo2"].value_counts(dropna=False))

print("\nMissing Open values in test:")
display(
    test_merged.loc[
        test_merged["Open"].isna(),
        ["Id", "Store", "Date", "DayOfWeek", "Promo", "StateHoliday"]
    ]
)

Promo2 distribution in training:
Promo2
1    509178
0    508031
Name: count, dtype: int64

Promo2 missingness check in training:


,Promo2SinceWeek,Promo2SinceYear,PromoInterval
Promo2,,,
0,508031,508031,508031
1,0,0,0



Promo2 distribution in test:
Promo2
1    23856
0    17232
Name: count, dtype: int64

Missing Open values in test:


,Id,Store,Date,DayOfWeek,Promo,StateHoliday
479,480,622,2015-09-17,4,1,0
1335,1336,622,2015-09-16,3,1,0
2191,2192,622,2015-09-15,2,1,0
3047,3048,622,2015-09-14,1,1,0
4759,4760,622,2015-09-12,6,0,0
5615,5616,622,2015-09-11,5,0,0
6471,6472,622,2015-09-10,4,0,0
7327,7328,622,2015-09-09,3,0,0
8183,8184,622,2015-09-08,2,0,0
9039,9040,622,2015-09-07,1,0,0


In [9]:
# Inspect Store 622 in the training dataset
store_622_train = train_merged[
    train_merged["Store"] == 622
].copy()

print("Store 622 training records:", len(store_622_train))

print("\nHistorical Open distribution:")
print(store_622_train["Open"].value_counts(dropna=False))

print("\nHistorical records around the end of training:")
display(
    store_622_train.sort_values("Date").tail(15)[
        ["Store", "Date", "DayOfWeek", "Open",
         "Promo", "StateHoliday", "Sales", "Customers"]
    ]
)

print("\nStore 622 test records:")
display(
    test_merged[test_merged["Store"] == 622].sort_values("Date")[
        ["Id", "Store", "Date", "DayOfWeek", "Open", "Promo"]
    ]
)

Store 622 training records: 942

Historical Open distribution:
Open
1    784
0    158
Name: count, dtype: int64

Historical records around the end of training:


,Store,Date,DayOfWeek,Open,Promo,StateHoliday,Sales,Customers
16231,622,2015-07-17,5,1,1,0,5263,462
15116,622,2015-07-18,6,1,0,0,2367,265
14001,622,2015-07-19,7,0,0,0,0,0
12886,622,2015-07-20,1,1,0,0,4688,429
11771,622,2015-07-21,2,1,0,0,3529,367
10656,622,2015-07-22,3,1,0,0,3721,376
9541,622,2015-07-23,4,1,0,0,3970,399
8426,622,2015-07-24,5,1,0,0,4183,440
7311,622,2015-07-25,6,1,0,0,2644,257
6196,622,2015-07-26,7,0,0,0,0,0



Store 622 test records:


,Id,Store,Date,DayOfWeek,Open,Promo
40711,40712,622,2015-08-01,6,1.0,0
39855,39856,622,2015-08-02,7,0.0,0
38999,39000,622,2015-08-03,1,1.0,1
38143,38144,622,2015-08-04,2,1.0,1
37287,37288,622,2015-08-05,3,1.0,1
36431,36432,622,2015-08-06,4,1.0,1
35575,35576,622,2015-08-07,5,1.0,1
34719,34720,622,2015-08-08,6,1.0,0
33863,33864,622,2015-08-09,7,0.0,0
33007,33008,622,2015-08-10,1,1.0,0


In [10]:
store_622_pattern = (
    train_merged.loc[
        train_merged["Store"] == 622
    ]
    .groupby("DayOfWeek")["Open"]
    .agg(["count", "mean", "min", "max"])
)

print("Store 622 historical opening pattern:")
display(store_622_pattern)

Store 622 historical opening pattern:


,count,mean,min,max
DayOfWeek,,,,
1,134,0.955224,0,1
2,135,0.992593,0,1
3,135,0.977778,0,1
4,135,0.940741,0,1
5,135,0.955556,0,1
6,134,1.000000,1,1
7,134,0.000000,0,0


In [11]:
# Remember which Open values were originally missing
test_merged["Open_was_missing"] = (
    test_merged["Open"].isna().astype("int8")
)

# Learn the most common opening status for each store and weekday
opening_pattern = (
    train_merged.dropna(subset=["Open"])
    .groupby(["Store", "DayOfWeek"])["Open"]
    .agg(lambda s: s.mode().min())
)

# Find rows where Open is missing
missing_mask = test_merged["Open"].isna()

# Look up the historical pattern
keys = pd.MultiIndex.from_frame(
    test_merged.loc[missing_mask, ["Store", "DayOfWeek"]]
)

inferred = opening_pattern.reindex(keys)

# Never silently leave missing values unresolved
assert inferred.notna().all(), (
    "Some missing Open values have no historical pattern."
)

# Impute only missing values
test_merged.loc[missing_mask, "Open"] = inferred.to_numpy()

print("Imputed rows:", int(missing_mask.sum()))
print("Remaining missing Open:", test_merged["Open"].isna().sum())

print("\nImputed records:")
display(
    test_merged.loc[
        test_merged["Open_was_missing"] == 1,
        ["Id", "Store", "Date", "DayOfWeek",
         "Open", "Open_was_missing"]
    ].sort_values("Date")
)

Imputed rows: 11
Remaining missing Open: 0

Imputed records:


,Id,Store,Date,DayOfWeek,Open,Open_was_missing
10751,10752,622,2015-09-05,6,1.0,1
9039,9040,622,2015-09-07,1,1.0,1
8183,8184,622,2015-09-08,2,1.0,1
7327,7328,622,2015-09-09,3,1.0,1
6471,6472,622,2015-09-10,4,1.0,1
5615,5616,622,2015-09-11,5,1.0,1
4759,4760,622,2015-09-12,6,1.0,1
3047,3048,622,2015-09-14,1,1.0,1
2191,2192,622,2015-09-15,2,1.0,1
1335,1336,622,2015-09-16,3,1.0,1


In [12]:
assert test_merged["Open"].isna().sum() == 0
assert set(test_merged["Open"].unique()).issubset({0, 1})

print("Open-value validation passed.")

Open-value validation passed.


In [13]:
competition_cols = [
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear"
]

for df in [train_merged, test_merged]:
    for col in competition_cols:
        df[f"{col}_was_missing"] = (
            df[col].isna().astype("int8")
        )

print("Missingness indicators created.")

Missingness indicators created.


In [14]:
for df in [train_merged, test_merged]:
    df["PromoInterval"] = df["PromoInterval"].fillna(
        "No_Promo2"
    )

print("Training PromoInterval missing:",
      train_merged["PromoInterval"].isna().sum())

print("Test PromoInterval missing:",
      test_merged["PromoInterval"].isna().sum())

Training PromoInterval missing: 0
Test PromoInterval missing: 0


In [15]:
print("Training CompetitionDistance:")
display(train_merged["CompetitionDistance"].describe())

print("\nTraining missing:",
      train_merged["CompetitionDistance"].isna().sum())

print("Test missing:",
      test_merged["CompetitionDistance"].isna().sum())

print("\nTraining missingness by store:")
display(
    train_merged.loc[
        train_merged["CompetitionDistance"].isna(),
        ["Store", "CompetitionDistance_was_missing"]
    ].drop_duplicates().head(10)
)

Training CompetitionDistance:


,CompetitionDistance
count,1.014567e+06
mean,5.430086e+03
std,7.715324e+03
min,2.000000e+01
25%,7.100000e+02
50%,2.330000e+03
75%,6.890000e+03
max,7.586000e+04



Training missing: 2642
Test missing: 96

Training missingness by store:


,Store,CompetitionDistance_was_missing
290,291,1
621,622,1
878,879,1


In [16]:
# Examine all stores with missing competition distance
missing_distance_stores = store.loc[
    store["CompetitionDistance"].isna(),
    ["Store", "CompetitionDistance",
     "CompetitionOpenSinceMonth",
     "CompetitionOpenSinceYear"]
]

print("Stores with missing CompetitionDistance:",
      len(missing_distance_stores))

display(missing_distance_stores)

Stores with missing CompetitionDistance: 3


,Store,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear
290,291,NaN,NaN,NaN
621,622,NaN,NaN,NaN
878,879,NaN,NaN,NaN


In [17]:
# Keep the merged data unchanged for reference
train_clean = train_merged.copy()
test_clean = test_merged.copy()

# Calculate the median from training data only
competition_median = train_clean["CompetitionDistance"].median()

print("Training median CompetitionDistance:", competition_median)

# Fill missing distances in both datasets using the training median
train_clean["CompetitionDistance"] = (
    train_clean["CompetitionDistance"].fillna(competition_median)
)

test_clean["CompetitionDistance"] = (
    test_clean["CompetitionDistance"].fillna(competition_median)
)

# Validate
print("\nRemaining missing values:")
print("Train:", train_clean["CompetitionDistance"].isna().sum())
print("Test:", test_clean["CompetitionDistance"].isna().sum())

assert train_clean["CompetitionDistance"].isna().sum() == 0
assert test_clean["CompetitionDistance"].isna().sum() == 0

print("\nValidation passed.")

Training median CompetitionDistance: 2330.0

Remaining missing values:
Train: 0
Test: 0

Validation passed.


In [18]:
def add_date_features(df):
    df = df.copy()

    # Parse Date without changing the original DayOfWeek column
    df["Date"] = pd.to_datetime(df["Date"], errors="raise")

    # Calendar features
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype("int16")

    # Rossmann convention: Monday=1, Sunday=7
    df["IsWeekend"] = df["DayOfWeek"].isin([6, 7]).astype("int8")

    # Position within the month
    df["MonthPeriod"] = pd.cut(
        df["Day"],
        bins=[0, 10, 20, 31],
        labels=["Beginning", "Middle", "End"],
        include_lowest=True
    )

    # Cyclical month features
    df["MonthSin"] = np.sin(2 * np.pi * df["Month"] / 12)
    df["MonthCos"] = np.cos(2 * np.pi * df["Month"] / 12)

    # Cyclical weekday features; Monday=1 through Sunday=7
    weekday_position = df["DayOfWeek"] - 1
    df["DayOfWeekSin"] = np.sin(2 * np.pi * weekday_position / 7)
    df["DayOfWeekCos"] = np.cos(2 * np.pi * weekday_position / 7)

    return df


train_clean = add_date_features(train_clean)
test_clean = add_date_features(test_clean)

print("Train shape:", train_clean.shape)
print("Test shape:", test_clean.shape)

print("\nNew date-feature preview:")
display(
    train_clean[
        [
            "Date", "DayOfWeek", "Year", "Month", "Day",
            "WeekOfYear", "IsWeekend", "MonthPeriod",
            "MonthSin", "MonthCos",
            "DayOfWeekSin", "DayOfWeekCos"
        ]
    ].head()
)

# Validation checks
assert train_clean["Date"].notna().all()
assert test_clean["Date"].notna().all()
assert train_clean["DayOfWeek"].between(1, 7).all()
assert test_clean["DayOfWeek"].between(1, 7).all()

print("\nDate-feature validation passed.")

Train shape: (1017209, 31)
Test shape: (41088, 31)

New date-feature preview:


,Date,DayOfWeek,Year,Month,Day,WeekOfYear,IsWeekend,MonthPeriod,MonthSin,MonthCos,DayOfWeekSin,DayOfWeekCos
0,2015-07-31,5,2015,7,31,31,0,End,-0.5,-0.866025,-0.433884,-0.900969
1,2015-07-31,5,2015,7,31,31,0,End,-0.5,-0.866025,-0.433884,-0.900969
2,2015-07-31,5,2015,7,31,31,0,End,-0.5,-0.866025,-0.433884,-0.900969
3,2015-07-31,5,2015,7,31,31,0,End,-0.5,-0.866025,-0.433884,-0.900969
4,2015-07-31,5,2015,7,31,31,0,End,-0.5,-0.866025,-0.433884,-0.900969



Date-feature validation passed.


In [19]:
# 1. Check dataset shapes
print("Train shape:", train_clean.shape)
print("Test shape:", test_clean.shape)

# 2. Check remaining missing values
print("\nTotal missing values:")
print("Train:", train_clean.isna().sum().sum())
print("Test:", test_clean.isna().sum().sum())

print("\nColumns with missing values in train:")
print(train_clean.isna().sum()[train_clean.isna().sum() > 0])

print("\nColumns with missing values in test:")
print(test_clean.isna().sum()[test_clean.isna().sum() > 0])

# 3. Check duplicate rows
print("\nExact duplicate rows:")
print("Train:", train_clean.duplicated().sum())
print("Test:", test_clean.duplicated().sum())

# 4. Check the date range
print("\nDate ranges:")
print("Train:", train_clean["Date"].min(), "to", train_clean["Date"].max())
print("Test:", test_clean["Date"].min(), "to", test_clean["Date"].max())

# 5. Check columns present in only one dataset
train_only = sorted(set(train_clean.columns) - set(test_clean.columns))
test_only = sorted(set(test_clean.columns) - set(train_clean.columns))

print("\nColumns only in train:", train_only)
print("Columns only in test:", test_only)

Train shape: (1017209, 31)
Test shape: (41088, 31)

Total missing values:
Train: 1662758
Test: 64896

Columns with missing values in train:
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2SinceWeek              508031
Promo2SinceYear              508031
dtype: int64

Columns with missing values in test:
CompetitionOpenSinceMonth    15216
CompetitionOpenSinceYear     15216
Promo2SinceWeek              17232
Promo2SinceYear              17232
dtype: int64

Exact duplicate rows:
Train: 0
Test: 0

Date ranges:
Train: 2013-01-01 00:00:00 to 2015-07-31 00:00:00
Test: 2015-08-01 00:00:00 to 2015-09-17 00:00:00

Columns only in train: ['Customers', 'Sales']
Columns only in test: ['Id', 'Open_was_missing']


In [20]:
# Keep feature columns consistent where possible
if "Open_was_missing" not in train_clean.columns:
    train_clean["Open_was_missing"] = 0

train_clean["Open_was_missing"] = (
    train_clean["Open_was_missing"].astype("int8")
)

test_clean["Open_was_missing"] = (
    test_clean["Open_was_missing"].astype("int8")
)

# Recheck the column differences
train_only = sorted(set(train_clean.columns) - set(test_clean.columns))
test_only = sorted(set(test_clean.columns) - set(train_clean.columns))

print("Columns only in train:", train_only)
print("Columns only in test:", test_only)

assert set(train_only) == {"Customers", "Sales"}
assert set(test_only) == {"Id"}

print("\nTrain/test column validation passed.")

Columns only in train: ['Customers', 'Sales']
Columns only in test: ['Id']

Train/test column validation passed.


In [21]:
from google.colab import files

# Save the preprocessed datasets
train_clean.to_csv("/content/train_preprocessed.csv", index=False)
test_clean.to_csv("/content/test_preprocessed.csv", index=False)

# Saving the files onto the system
files.download("/content/train_preprocessed.csv")
files.download("/content/test_preprocessed.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>